# 使用 Realtime Mini 进行构建

从小我就对 Jarvis 的设想着迷——一个能够自主处理复杂工作流的智能助手。那时我没有意识到，我想象的其实就是语音智能体的未来。OpenAI 率先通过推出 `4o-audio` 让这一愿景成为现实，最近又发布了 [GPT Realtime Mini](https://platform.openai.com/docs/models/gpt-realtime-mini)，将成本降低了 70%，使其更加普及，同时带来了更低的延迟以及工具调用方面的重大改进。

然而，使用语音模型进行构建与使用纯文本界面有着本质的不同。除了提示工程之外，音频模型还带来了新的挑战：它们对延迟更加敏感，需要管理 WebRTC 会话，并且语音活动检测（VAD）还会引入额外的不确定性。

为了简化这一过程，OpenAI 发布了 Python 和 TypeScript 两个版本的 Agents SDK，并提供了详细的示例，展示我们推荐的构建可靠语音智能体的设计模式。

在深入代码之前，我们先明确要构建的究竟是什么，以及它如何融入更广泛的智能体移交（handoff）架构。

## 系统架构
今天，我们将使用 **“移交架构”** 构建一个极其简单的客户支持应用。**“移交架构”** 是指由一个 **主智能体** 充当所有传入客户查询的编排者。主智能体不会直接处理每个请求，而是分析用户消息背后的意图，并将其 **归类到 2 条核心路径之一**：

1. 一般问题和基础支持（无需身份验证）。
2. 具体问题（在执行查询前需要对用户进行身份验证）。

根据这一分类，主智能体会将 **对话移交** 给专门为该特定任务设计的专家智能体。

![替代文本](../../images/byo_realtime_diagram.png)

## 设置
我们不会从零开始，而是基于 [openai-agents-js](https://github.com/openai/openai-agents-js/tree/main) 仓库进行开发，所以先克隆仓库、安装必要的依赖，并构建 Web 演示
```bash
git clone https://github.com/openai/openai-agents-js/tree/main
```

克隆完成后，按照 readme 中的步骤开始
```bash
npm install @openai/agents zod@3
pnpm examples:realtime-next
```

如果一切正常，你应该会看到一个简单的聊天界面
![替代文本](../../images/byo_realtime_starting.png)

## 主智能体
很好！现在我们已经克隆了仓库，接下来将修改 `openai-agents-js/examples/realtime-next/src/app/page.tsx`，先从 **主智能体** 开始。**主智能体** 是整个应用栈的入口点。它充当任何用户查询的意图分类器，决定如何在不同层之间进行路由。

其实现相当直接
```js
const mainAgent = new RealtimeAgent({
  name: 'Main Agent',
  instructions:
    'You are the entry point for all customer queries. Default to the no-auth QA flow. If authentication is needed and validated, escalate to the Auth Layer by handing off to either the Flight Status Checker or Rebooking Agent. Do not answer policy questions from your own knowledge; rely on subordinate agents and tools.',
  tools: [
    checkFlightsTool,
  ],
  handoffs: [qaAgent],
});
```

## QA 智能体

现在我们已经构建了主智能体，下一步是添加一个专门的辅助智能体来处理某一类客户查询。对于一般性的航空公司政策问题，这就是 QA 智能体。

在真实的产品中，这个智能体会支撑更复杂的体验：它会摄取公司特定的 PDF 和其他参考资料，对其进行嵌入，并在运行时动态查询这些文档，以提供准确、基于政策的答案。

```
┌────────────┐      ┌────────────┐      ┌────────────────────────┐      ┌────────────┐
│ User Query │ ───► │ QA Agent   │ ───► │ Vector DB / Retriever  │ ───► │ LLM Answer │
└────────────┘      └────────────┘      └────────────────────────┘      └────────────┘
                          │                     │
                          │ build search        │ top-k context
                          ▼                     ▼
                 (semantic search)      (grounded generation)

```

这通常需要构建一个完整的向量数据库服务，对客户的查询进行嵌入并检索最相关的结果。为了让本演示保持简单，我们将模拟流水线中的这一部分。

如果你有兴趣了解如何实现功能完备的检索系统，请查看我们关于该主题的其他 cookbook，[点击这里](https://cookbook.openai.com/examples/vector_databases/pinecone/readme)。

```js
const documentLookupTool = tool({
  name: 'document_lookup_tool',
  description: 'Looks up answers from known airline documentation to handle general questions without authentication.',
  parameters: z.object({
    request: z.string(),
  }),
  execute: async ({ request }) => {
    const mockDocument = `**Airline Customer Support — Quick Reference**

1. Each passenger may bring 1 carry-on (22 x 14 x 9) and 1 personal item.
2. Checked bags must be under 50 lbs; overweight fees apply.
3. Online check-in opens 24 hours before departure.
4. Seat upgrades can be requested up to 1 hour before boarding.
5. Wi‑Fi is complimentary on all flights over 2 hours.
6. Customers can change flights once for free within 24 hours of booking.
7. Exit rows offer extra legroom and require passengers to meet safety criteria.
8. Refunds can be requested for canceled or delayed flights exceeding 3 hours.
9. Pets are allowed in the cabin if under 20 lbs and in an approved carrier.
10. For additional help, contact our support team via chat or call center.`;
    return mockDocument;
  },
});
```

与之前定义主智能体时一样，我们将创建另一个 `RealtimeAgent` 实例，但这一次我们会提供一个 `documentLookupTool`。

```js
const qaAgent = new RealtimeAgent({
  name: 'QA Agent',
  instructions:
    'You handle general customer questions using the document lookup tool. Use only the document lookup for answers. If the request may involve personal data or operations (rebooking, flight status), call the auth check tool. If auth is required and validated, handoff to the appropriate Auth Layer agent.',
  tools: [documentLookupTool],
});
```

## 航班状态智能体
我们已经打下了强大的基础：一个能够处理入站客户查询的主智能体，以及一个搜索文档库以提供准确、基于政策的答案的 QA 智能体。

目前还缺少的是一个处理客户特定信息的层——例如“我的航班状态如何？”或“我应该去哪个航站楼？”之类的查询。为了支持这类个性化交互，我们需要在工作流中嵌入一个身份验证层，使系统能够安全地访问用户特定的数据并据此作答。

```
┌────────────┐      ┌──────────────┐      ┌───────────────────────┐      ┌───────────────────────┐
│ User Query │ ───► │ Auth Layer   │ ───► │ Customer Data Access  │ ───► │ LLM Answer (Personal) │
└────────────┘      └──────────────┘      └───────────────────────┘      └───────────────────────┘
                        │                          │
                        │ verify identity          │ query flight / account
                        ▼                          ▼
                (token, SSO, OTP, etc.)   (e.g., flight status, profile info)
```
幸运的是，Agents SDK 正是为支持这类用例而设计的。对于涉及敏感账户级信息的客户支持场景，我们可以在 `tool` 中使用 `needsApproval` 参数来确保适当的访问控制，它要求用户在访问任何受保护数据之前先进行身份验证。

```js
const checkFlightsTool = tool({
  name: 'checkFlightsTool',
  description: 'Call this tool if the user queries about their current flight status',
  parameters: z.object({}),
  // Require approval so the UI can collect creds before executing.
  needsApproval: true,
  execute: async () => {
    if (!credState.username || !credState.password) {
      return 'Authentication missing.';
    }
    return `${credState.username} you are currently booked on the 8am flight from SFO to JFK`;
  },
});
```

当工具注册时带有 `needsApproval`，它会在会话期间自动发出 `tool_approval_requested` 事件。这使我们可以在 Web 应用的 `RealtimeAgent` 实例化代码块中添加逻辑来监听这些事件，并相应地更新 UI——例如，在继续之前提示用户批准或进行身份验证。

```js
  const [credUsername, setCredUsername] = useState('');
  const [credPassword, setCredPassword] = useState('');
  const [pendingApproval, setPendingApproval] = useState<any | null>(null);

  useEffect(() => {
    session.current = new RealtimeSession(mainAgent, {
        // other configs go here! 
    });
    // various other event based logic goes here!
    session.current.on(
      'tool_approval_requested',
      (_context, _agent, approvalRequest) => {
        setPendingApproval(approvalRequest.approvalItem); // <- Alterations to react state!
        setCredUsername('');
        setCredPassword('');
        setCredOpen(true);
      },
    );
  }, []);
  // ....
  return (
    {credOpen && (
        <div className="fixed inset-0 z-50">
          // ... remainder of component logic
        </div>
      )}
  )
```

## 最终代码片段
这样，我们就完成了！你现在已经构建了客户支持应用的核心组件：

* 一个能够处理各类客户支持查询的通用智能体
* 一个验证用户身份并检索客户特定信息的身份验证工作流

一切就绪后，`realtime-next/src/app/page.tsx` 的最终版本应如下所示。

```js
'use client';

import {
  RealtimeAgent,
  RealtimeSession,
  tool,
  TransportEvent,
  RealtimeOutputGuardrail,
  OutputGuardrailTripwireTriggered,
  RealtimeItem,
} from '@openai/agents/realtime';
import { useEffect, useRef, useState } from 'react';
import { z } from 'zod';
import { getToken } from './server/token.action';
import { App } from '@/components/App';
import { CameraCapture } from '@/components/CameraCapture';

// Demo-only credential store the tool can read at execution time
const credState: { username?: string; password?: string } = {};

// ---------------------------------------------
// Tools.

const documentLookupTool = tool({
  name: 'document_lookup_tool',
  description: 'Looks up answers from known airline documentation to handle general questions without authentication.',
  parameters: z.object({
    request: z.string(),
  }),
  execute: async ({ request }) => {
    const mockDocument = `**Airline Customer Support — Quick Reference**

1. Each passenger may bring 1 carry-on (22 x 14 x 9) and 1 personal item.
2. Checked bags must be under 50 lbs; overweight fees apply.
3. Online check-in opens 24 hours before departure.
4. Seat upgrades can be requested up to 1 hour before boarding.
5. Wi‑Fi is complimentary on all flights over 2 hours.
6. Customers can change flights once for free within 24 hours of booking.
7. Exit rows offer extra legroom and require passengers to meet safety criteria.
8. Refunds can be requested for canceled or delayed flights exceeding 3 hours.
9. Pets are allowed in the cabin if under 20 lbs and in an approved carrier.
10. For additional help, contact our support team via chat or call center.`;
    return mockDocument;
  },
});

const checkFlightsTool = tool({
  name: 'checkFlightsTool',
  description: 'Call this tool if the user queries about their current flight status',
  parameters: z.object({}),
  // Require approval so the UI can collect creds before executing.
  needsApproval: true,
  execute: async () => {
    if (!credState.username || !credState.password) {
      return 'Authentication missing.';
    }
    return `${credState.username} you are currently booked on the 8am flight from SFO to JFK`;
  },
});

// ---------------------------------------------
// Agents for each layer.

// 2. No-Auth Layer: QA Agent with doc lookup and auth check tool.
const qaAgent = new RealtimeAgent({
  name: 'QA Agent',
  instructions:
    'You handle general customer questions using the document lookup tool. Use only the document lookup for answers. If the request may involve personal data or operations (rebooking, flight status), call the auth check tool. If auth is required and validated, handoff to the appropriate Auth Layer agent.',
  tools: [documentLookupTool],
});

// 1. Main Agent: entry point and routing.
const mainAgent = new RealtimeAgent({
  name: 'Main Agent',
  instructions:
    'You are the entry point for all customer queries. Default to the no-auth QA flow. If authentication is needed and validated, escalate to the Auth Layer by handing off to either the Flight Status Checker or Rebooking Agent. Do not answer policy questions from your own knowledge; rely on subordinate agents and tools.',
  tools: [
    checkFlightsTool,
  ],
  handoffs: [qaAgent],
});

// Cross-handoffs so agents can return or escalate.
qaAgent.handoffs = [mainAgent];

export default function Home() {
  const session = useRef<RealtimeSession<any> | null>(null);
  const [isConnected, setIsConnected] = useState(false);
  const [isMuted, setIsMuted] = useState(false);
  const [outputGuardrailResult, setOutputGuardrailResult] =
    useState<OutputGuardrailTripwireTriggered<any> | null>(null);

  const [events, setEvents] = useState<TransportEvent[]>([]);
  const [history, setHistory] = useState<RealtimeItem[]>([]);
  const [mcpTools, setMcpTools] = useState<string[]>([]);
  const [credOpen, setCredOpen] = useState(false);
  const [credUsername, setCredUsername] = useState('');
  const [credPassword, setCredPassword] = useState('');
  const [pendingApproval, setPendingApproval] = useState<any | null>(null);

  useEffect(() => {
    session.current = new RealtimeSession(mainAgent, {
      model: 'gpt-realtime-mini',
      outputGuardrailSettings: {
        debounceTextLength: 200,
      },
      config: {
        audio: {
          output: {
            voice: 'cedar',
          },
        },
      },
    });
    session.current.on('transport_event', (event) => {
      setEvents((events) => [...events, event]);
    });
    session.current.on('mcp_tools_changed', (tools) => {
      setMcpTools(tools.map((t) => t.name));
    });
    session.current.on(
      'guardrail_tripped',
      (_context, _agent, guardrailError) => {
        setOutputGuardrailResult(guardrailError);
      },
    );
    session.current.on('history_updated', (history) => {
      setHistory(history);
    });
    session.current.on(
      'tool_approval_requested',
      (_context, _agent, approvalRequest) => {
        setPendingApproval(approvalRequest.approvalItem);
        setCredUsername('');
        setCredPassword('');
        setCredOpen(true);
      },
    );
  }, []);

  async function connect() {
    if (isConnected) {
      await session.current?.close();
      setIsConnected(false);
    } else {
      const token = await getToken();
      try {
        await session.current?.connect({
          apiKey: token,
        });
        setIsConnected(true);
      } catch (error) {
        console.error('Error connecting to session', error);
      }
    }
  }

  async function toggleMute() {
    if (isMuted) {
      await session.current?.mute(false);
      setIsMuted(false);
    } else {
      await session.current?.mute(true);
      setIsMuted(true);
    }
  }

  function handleCredCancel() {
    const approval = pendingApproval;
    setCredOpen(false);
    setPendingApproval(null);
    if (approval) session.current?.reject(approval);
  }

  function handleCredSubmit(e: React.FormEvent) {
    e.preventDefault();
    if (!credUsername || !credPassword) return;
    // Store creds for the tool to read
    credState.username = credUsername;
    credState.password = credPassword;
    const approval = pendingApproval;
    setCredOpen(false);
    setPendingApproval(null);
    setCredUsername('');
    setCredPassword('');
    if (approval) session.current?.approve(approval);
  }

  return (
    <div className="relative">
      {credOpen && (
        <div className="fixed inset-0 z-50">
          <div className="absolute inset-0 bg-black/50" />
          <div className="fixed top-0 left-0 right-0 flex justify-center p-4">
            <form
              onSubmit={handleCredSubmit}
              className="w-full max-w-sm rounded-lg bg-white p-4 shadow-xl"
            >
              <div className="mb-2 text-sm font-semibold">Authentication Required</div>
              <div className="mb-3 text-xs text-gray-600">
                Enter username and password to continue.
              </div>
              <input
                className="mb-2 w-full rounded border border-gray-300 p-2 text-sm focus:border-gray-500 focus:outline-none"
                placeholder="Username"
                value={credUsername}
                onChange={(e) => setCredUsername(e.target.value)}
              />
              <input
                type="password"
                className="mb-3 w-full rounded border border-gray-300 p-2 text-sm focus:border-gray-500 focus:outline-none"
                placeholder="Password"
                value={credPassword}
                onChange={(e) => setCredPassword(e.target.value)}
              />
              <div className="flex justify-end gap-2">
                <button
                  type="button"
                  className="rounded bg-gray-100 px-3 py-1.5 text-sm hover:bg-gray-200"
                  onClick={handleCredCancel}
                >
                  Cancel
                </button>
                <button
                  type="submit"
                  className="rounded bg-black px-3 py-1.5 text-sm text-white hover:bg-gray-800 disabled:opacity-50"
                  disabled={!credUsername || !credPassword}
                >
                  Continue
                </button>
              </div>
            </form>
          </div>
        </div>
      )}
      <App
        isConnected={isConnected}
        isMuted={isMuted}
        toggleMute={toggleMute}
        connect={connect}
        history={history}
        outputGuardrailResult={outputGuardrailResult}
        events={events}
        mcpTools={mcpTools}
      />
      <div className="fixed bottom-4 right-4 z-50">
        <CameraCapture
          disabled={!isConnected}
          onCapture={(dataUrl) => {
            if (!session.current) return;
            session.current.addImage(dataUrl, { triggerResponse: false });
          }}
        />
      </div>
    </div>
  );
}
```